# Gemini vs Open-Source LLMs (Mistral-7B, Phi-2, TinyLlama) on the SDF Document

**Document:** `sample-sdf-document.pdf` (Cytiva storage-conditions letter + two Certificates of Quality, 3 pages)
**Stack:** PyMuPDF + LlamaIndex + BGE-small embeddings, with four interchangeable LLMs
**Author:** Bashar

Retrieval is identical for every model: the same chunks, in the same order, are put into the same prompt. Only the LLM that writes the answer changes, so differences in the answers come from the models.

| Model | Size | Runs | Context window used |
|---|---|---|---|
| Gemini (`gemini-3.8-flash`) | not published | Google's servers, API key | very large |
| Mistral-7B-Instruct v0.2 | 7.2B parameters, 4.4 GB file | locally, llama.cpp | 4,096 (supports 32k) |
| Phi-2 | 2.7B parameters, 1.8 GB file | locally, llama.cpp | 2,048 (its maximum) |
| TinyLlama-1.1B-Chat | 1.1B parameters, 0.7 GB file | locally, llama.cpp | 2,048 (its maximum) |

Three experiments, as set out in the task: **speed**, **accuracy**, and **context limit**.

**Before running:** Runtime → Change runtime type → **T4 GPU**, and add a Colab secret named `GOOGLE_API_KEY`.

## Step 0: Install dependencies

A prebuilt CUDA build of `llama-cpp-python` 0.3.x is installed first. (The course's 0.2.90 gets replaced by a slow CPU-only source build when `llama-index-llms-llama-cpp` is installed afterwards, because that package now needs 0.3 or newer.)

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf llama-index-core llama-index-llms-llama-cpp llama-index-llms-google-genai llama-index-embeddings-huggingface google-genai huggingface_hub nest_asyncio

In [1]:
import nest_asyncio
nest_asyncio.apply()

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU offload supported:", llama_cpp.llama_supports_gpu_offload())

llama-cpp-python 0.3.36 | GPU offload supported: False


## Step 1: Gemini key, model files and the PDF

In [2]:
import os, glob

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
assert GOOGLE_API_KEY, "Add a Colab secret named GOOGLE_API_KEY"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

MODELS_DIR = os.environ.get("MODELS_DIR", "/content/models")
GGUF_FILES = {
    "Mistral-7B": ("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    "Phi-2": ("TheBloke/phi-2-GGUF", "phi-2.Q4_K_M.gguf"),
    "TinyLlama": ("TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF", "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"),
}
from huggingface_hub import hf_hub_download
for name, (repo, filename) in GGUF_FILES.items():
    path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(path):
        hf_hub_download(repo, filename, local_dir=MODELS_DIR)
    print(f"{name:<11} {os.path.getsize(path) / 1024**3:.2f} GB")

candidates = glob.glob("*sdf-document*.pdf") or glob.glob("/content/*sdf-document*.pdf")
if not candidates:
    from google.colab import files
    print("Please select 'sample-sdf-document.pdf' to upload:")
    candidates = list(files.upload().keys())
pdf_path = candidates[0]
print("Using PDF:", pdf_path)

Mistral-7B  4.07 GB
Phi-2       1.67 GB
TinyLlama   0.62 GB
Using PDF: sample-sdf-document.pdf


## Step 2: Retrieval (shared by all four models)

Same index as the earlier tasks: one `Document` per page, 128-token chunks with 20 overlap, BGE-small with its query prefix. `top_k=5`, a little wider than before, because one of the task's questions asks for a summary of the whole document.

Every model gets exactly the same prompt, built here rather than by LlamaIndex's response synthesizer, so it can be printed and checked. Each chunk is labelled with its page number.

In [3]:
import fitz  # PyMuPDF
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

TOP_K = 5
MAX_NEW_TOKENS = 384

def load_pdf_with_pymupdf(path):
    # Parse a PDF into one LlamaIndex Document per page.
    doc = fitz.open(path)
    documents = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if not text.strip():
            continue
        documents.append(Document(text=text, metadata={"page_number": i + 1},
                                  excluded_embed_metadata_keys=["page_number"]))
    doc.close()
    return documents

Settings.llm = None
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")
nodes = SentenceSplitter(chunk_size=128, chunk_overlap=20).get_nodes_from_documents(load_pdf_with_pymupdf(pdf_path))
index = VectorStoreIndex(nodes)

PROMPT = (
    "Context information from a pharmaceutical quality document is below.\n"
    "---------------------\n{context}\n---------------------\n"
    "Using only the context above, answer the question. "
    "If the context does not contain the answer, say so.\n"
    "Question: {question}\nAnswer:"
)

def build_prompt(question, chunks):
    context = "\n\n".join(f"[Page {n.metadata['page_number']}]\n{n.get_content()}" for n in chunks)
    return PROMPT.format(context=context, question=question)

QUERIES = [  # the three queries suggested in the task
    "What sterilization method was used for this product?",
    "Summarize the key specifications in this document.",
    "What are the storage conditions for this product?",
]
retrieved = {q: index.as_retriever(similarity_top_k=TOP_K).retrieve(q) for q in QUERIES}
for q in QUERIES:
    print(f"{q}\n  pages retrieved: {[n.metadata['page_number'] for n in retrieved[q]]}")

LLM is explicitly disabled. Using MockLLM.


What sterilization method was used for this product?
  pages retrieved: [2, 2, 3, 3, 2]
Summarize the key specifications in this document.
  pages retrieved: [3, 2, 2, 2, 1]
What are the storage conditions for this product?
  pages retrieved: [1, 1, 2, 3, 2]


## Step 3: Accuracy checks

Two automatic checks, so accuracy is not judged only by eye:

- **Key facts:** for the two factual questions, the facts a correct answer must contain (from reading the document).
- **Numbers not in the context:** every number in an answer is looked up in the retrieved text. A number that is not there was made up or garbled. This catches invented values, but not a real number attached to the wrong product or field, so the answers are also read by hand (see Findings).

The summary question has no single right answer, so it is judged by reading the answers (see Findings).

In [4]:
import re

KEY_FACTS = {
    QUERIES[0]: {"Autoclave": r"autoclav", "121 °C": r"121", "Gamma irradiation": r"gamma",
                 "25-40 kGy": r"25(\.0)?\s*(kGy)?\s*(-|–|to|and)\s*40"},
    QUERIES[2]: {"Above +5 °C": r"(?<![\d.])\+?\s?5\s?°?\s?C\b", "Operating +2 to +40 °C": r"\+?2\s?°?\s?C?\s*(-|–|to|and)\s*\+?40",
                 "Brittleness / cracking risk": r"brittle|crack"},
}

def numbers_in(text):
    text = re.sub(r"(?m)^\s*\d+\.\s", " ", text)          # ignore list numbering like "1. "
    return set(re.findall(r"\d+(?:\.\d+)?", text))

def unsupported_numbers(answer, chunks):
    context = " ".join(n.get_content() for n in chunks)
    allowed = numbers_in(context) | numbers_in(context.replace(" ", ""))  # "29 1846 12" == "29184612"
    allowed |= {str(n.metadata["page_number"]) for n in chunks}           # "[Page 3]" labels are in the prompt too
    return sorted(numbers_in(answer) - allowed, key=float)

def facts_found(question, answer):
    facts = KEY_FACTS.get(question)
    if not facts:
        return None
    return [name for name, pattern in facts.items() if re.search(pattern, answer, re.I)]

## Step 4: The models

Each local model is given the prompt format it was trained on. Without this, small models in particular tend to continue the prompt instead of answering it. Temperature is 0.1 for every model, and the local models may write up to 384 tokens.

Gemini's output limit is left at its default. Gemini 3 "thinks" before it answers, and those hidden thinking tokens count against the same limit: in a test, a 50-token cap made even "Say hello in five words" fail, because Gemini used 437 thinking tokens to write a 7-token reply. The thinking tokens are recorded below because they cost time and money even though nobody sees them.

In [5]:
import gc, time
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.google_genai import GoogleGenAI

N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0

PROMPT_FORMATS = {
    # Mistral-Instruct: llama.cpp adds the <s> start token itself
    "Mistral-7B": (lambda p, system_prompt=None: f"[INST] {p.strip()} [/INST]", []),
    # Phi-2 is a base model; this is the QA format from its model card. It also echoes the prompt's
    # "Question:" label and starts inventing new questions, so that is a stop sequence too
    "Phi-2": (lambda p, system_prompt=None: f"Instruct: {p.strip()}\nOutput:", ["Instruct:", "<|endoftext|>", "Question:"]),
    # TinyLlama-Chat uses the Zephyr chat format
    "TinyLlama": (lambda p, system_prompt=None: f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{p.strip()}</s>\n<|assistant|>\n", ["</s>"]),
}
CONTEXT_WINDOWS = {"Mistral-7B": 4096, "Phi-2": 2048, "TinyLlama": 2048}

def load_local(name):
    fmt, stop = PROMPT_FORMATS[name]
    return LlamaCPP(
        model_path=os.path.join(MODELS_DIR, GGUF_FILES[name][1]),
        temperature=0.1, max_new_tokens=MAX_NEW_TOKENS, context_window=CONTEXT_WINDOWS[name],
        completion_to_prompt=fmt, generate_kwargs={"stop": stop, "seed": 42},
        model_kwargs={"n_gpu_layers": N_GPU_LAYERS}, verbose=False)

def load_gemini():
    return GoogleGenAI(model="models/gemini-3.8-flash", temperature=0.1)

def with_retries(fn, retries=5):
    # Gemini's free tier occasionally returns 503 "high demand"; wait and retry instead of failing the run
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1 or "MAX_TOKENS" in str(e):
                raise
            print(f"  call failed ({type(e).__name__}); retrying in {10 * (attempt + 1)}s")
            time.sleep(10 * (attempt + 1))

def generate(llm, prompt):
    # Returns the answer plus timing, token counts and whether it stopped at the output limit
    t0 = time.perf_counter()
    try:
        resp = with_retries(lambda: llm.complete(prompt))
    except RuntimeError as e:
        if "MAX_TOKENS" not in str(e):
            raise
        # LlamaIndex's Gemini wrapper raises instead of returning a truncated answer
        return {"answer": "(no answer: hit the output token limit)", "seconds": time.perf_counter() - t0,
                "prompt_tokens": None, "answer_tokens": None, "thinking_tokens": None, "cut": True}
    seconds = time.perf_counter() - t0
    raw = resp.raw
    if "choices" in raw:                                   # llama.cpp
        usage = raw["usage"]
        return {"answer": resp.text.strip(), "seconds": seconds, "prompt_tokens": usage["prompt_tokens"],
                "answer_tokens": usage["completion_tokens"], "thinking_tokens": 0,
                "cut": raw["choices"][0]["finish_reason"] == "length"}
    usage = raw["usage_metadata"]                          # Gemini
    return {"answer": resp.text.strip(), "seconds": seconds, "prompt_tokens": usage["prompt_token_count"],
            "answer_tokens": usage["candidates_token_count"], "thinking_tokens": usage.get("thoughts_token_count") or 0,
            "cut": "MAX_TOKENS" in str(raw.get("finish_reason"))}

## Step 5: Speed and accuracy run

Each model answers the three questions. Local models are loaded one at a time and freed afterwards. Load time is reported separately and not counted in the per-answer speed.

In [6]:
results = []

def run_model(name, llm):
    for q in QUERIES:
        g = generate(llm, build_prompt(q, retrieved[q]))
        results.append({"model": name, "query": q, **g,
                        "facts": facts_found(q, g["answer"]), "unsupported": unsupported_numbers(g["answer"], retrieved[q])})
        thinking = f", {g['thinking_tokens']} hidden thinking tokens" if g["thinking_tokens"] else ""
        print(f"\n--- {name} | {q}\n({g['seconds']:.1f}s, {g['answer_tokens']} answer tokens{thinking}"
              f"{', CUT OFF at token limit' if g['cut'] else ''})\n{g['answer']}")

print("=" * 100 + "\nGemini\n" + "=" * 100)
run_model("Gemini", load_gemini())

load_times = {}
for name in ["Mistral-7B", "Phi-2", "TinyLlama"]:
    print("\n" + "=" * 100 + f"\n{name}\n" + "=" * 100)
    t0 = time.perf_counter()
    llm = load_local(name)
    load_times[name] = time.perf_counter() - t0
    print(f"loaded in {load_times[name]:.1f}s on {'GPU' if N_GPU_LAYERS else 'CPU'}")
    run_model(name, llm)
    del llm
    gc.collect()

Gemini



--- Gemini | What sterilization method was used for this product?
(19.5s, 59 answer tokens, 357 hidden thinking tokens)
Based on the provided context, two sterilization methods are listed:

* **Autoclave** (121°C for > 15 min) for the pump tubing
* **Gamma Irradiation** (25.0 – 40.0 kGy) for the inlets



--- Gemini | Summarize the key specifications in this document.
(11.5s, 340 answer tokens, 852 hidden thinking tokens)
Based on the provided document, the key specifications are:

* **Operating Temperature:** 
  * +2°C to +40°C (for High Flow Kit F, Part #29477427, and High Flow Gradient C, Part #29184612).

* **Processing and Sterilization Limits:**
  * **Autoclave (Pump tubing):** 121°C for > 15 min.
  * **Gamma Irradiation (Inlets):** 25.0 – 40.0 kGy (also listed as 0 – 40.0 kGy, Conforms).

* **Testing and Inspection Criteria:**
  * **Flow Rate Test:** Per SOP-QC-042 (Pass).
  * **Pressure Integrity:** Max 0.5 bar, 5 min hold (Pass).
  * **Visual Inspection:** No defects visible (Pass).
  * **Package Integrity:** Sealed, no damage (Pass).

* **Regulatory Conformance and Bio-Safety:**
  * **Bio-safety:** All polymeric materials in contact with process fluid comply with USP <88> Biological Reactivity Test Class VI.
  * **Animal-Derived Ingredients:** Free from animal-derived ingredi


--- Gemini | What are the storage conditions for this product?
(42.8s, 57 answer tokens, 353 hidden thinking tokens)
Based on the provided document, the recommended storage temperature for the ÄKTA ready flow kits (both standard and modified) is **> +5 °C**. 

The document notes that extended storage below +5 °C could lead to brittleness or cracking of the plastic connectors.

Mistral-7B


loaded in 4.1s on CPU


...


TinyLlama


loaded in 0.6s on CPU



--- TinyLlama | What sterilization method was used for this product?
(5.7s, 93 answer tokens)
The context does not contain the answer to the question as it only mentions that the product was manufactured to meet its specification and has been verified to meet predefined Critical to Quality attributes. However, the context mentions that the flow kit, AKTA ready, has been formally relased for delivery, indicating that it has been formally relased for delivery. Therefore, the sterilization method used for this product is not specified in the context.



--- TinyLlama | Summarize the key specifications in this document.
(8.2s, 186 answer tokens)
The key specification in this document is the flow rate test for the product, which is performed at 40.0 kGy and for a pressure integrity test that includes pressure integration, a maximum 0.5 bar hold, and a pass for no defects visible. The product is also biocompatible, biologically safe, and has a USP Class VI biocompatibility rating. The product is manufactured in accordance with ISO 9001 certification and has a validated quality assurance system. The product is also validated for its compliance with the USP Class VI biocompatibility rating and is free from animal-derived ingredients or in compliance with EM-410/01. The product is also certified by Cytiva Westborough Quality Assurance and is validated for its compliance with ISO 9001 certification.



--- TinyLlama | What are the storage conditions for this product?
(5.4s, 80 answer tokens)
The context does not contain the answer to the question as it only provides information on the recommended storage temperature for standard ÄKTA ready flow kits, which is specified as > +5 C. The context also mentions that the recommended storage temperature for modified ÄKTA ready flow kits is specified as > +5 C, but does not provide any information on the storage conditions for these kits.


## Step 6: Scorecard

In [7]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

df = pd.DataFrame(results)
def fact_score(r):
    total = len(KEY_FACTS.get(r["query"], {}))
    return f"{len(r['facts'])}/{total}" if total else "n/a"

scorecard = pd.DataFrame({
    "Model": df["model"],
    "Query": df["query"].str.slice(0, 32),
    "Key facts": df.apply(fact_score, axis=1),
    "Numbers not in context": df["unsupported"].apply(lambda u: ", ".join(u) or "none"),
    "Answer tokens": df["answer_tokens"],
    "Thinking tokens": df["thinking_tokens"],
    "Seconds": df["seconds"].round(1),
    "Cut off": df["cut"].map({True: "yes", False: "no"}),
})
scorecard

,Model,Query,Key facts,Numbers not in context,Answer tokens,Thinking tokens,Seconds,Cut off
0,Gemini,What sterilization method was us,4/4,none,59,357,19.5,no
1,Gemini,Summarize the key specifications,n/a,none,340,852,11.5,no
2,Gemini,What are the storage conditions,2/3,none,57,353,42.8,no
3,Mistral-7B,What sterilization method was us,4/4,none,74,0,24.7,no
4,Mistral-7B,Summarize the key specifications,n/a,none,193,0,42.0,no
5,Mistral-7B,What are the storage conditions,2/3,none,115,0,32.0,no
6,Phi-2,What sterilization method was us,1/4,none,10,0,6.5,no
7,Phi-2,Summarize the key specifications,n/a,none,384,0,31.6,yes
8,Phi-2,What are the storage conditions,1/3,none,384,0,31.6,yes
9,TinyLlama,What sterilization method was us,0/4,none,93,0,5.7,no


In [8]:
speed = df.groupby("model", sort=False).agg(
    avg_seconds=("seconds", "mean"), avg_answer_tokens=("answer_tokens", "mean"), avg_thinking_tokens=("thinking_tokens", "mean"))
# tokens generated per second, counting Gemini's hidden thinking tokens as generated work
speed["tokens_per_s"] = ((speed["avg_answer_tokens"] + speed["avg_thinking_tokens"]) / speed["avg_seconds"]).round(1)
speed["load_s"] = pd.Series(load_times).round(1)
speed.round(1)

,avg_seconds,avg_answer_tokens,avg_thinking_tokens,tokens_per_s,load_s
model,,,,,
Gemini,24.6,152.0,520.7,27.4,NaN
Mistral-7B,32.9,127.3,0.0,3.9,4.1
Phi-2,23.2,259.3,0.0,11.2,1.2
TinyLlama,6.4,119.7,0.0,18.6,0.6


## Step 7: Context-limit test

The task asks whether a model can handle long responses or cuts them off. Here each model gets the **whole document** (all 12 chunks) and is asked for a detailed summary with room for up to 1,024 new tokens.

For the local models the context window is a hard ceiling: prompt and answer together must fit in it, and llama.cpp stops generating when the window is full. Phi-2 and TinyLlama only have 2,048 tokens in total.

In [9]:
LONG_QUESTION = "Give a detailed summary of everything in this document, covering every product, specification, test and date."
long_prompt = build_prompt(LONG_QUESTION, nodes)
LONG_MAX_TOKENS = 1024

long_results = []
def run_long(name, llm, window):
    g = generate(llm, long_prompt)
    long_results.append({"Model": name, "Context window": window, "Prompt tokens": g["prompt_tokens"],
                         "Room left for answer": (window - g["prompt_tokens"]) if isinstance(window, int) else "n/a",
                         "Answer tokens": g["answer_tokens"], "Stopped at limit": "yes" if g["cut"] else "no",
                         "Seconds": round(g["seconds"], 1), "Ending": g["answer"][-120:].replace("\n", " ")})
    print(f"\n--- {name}: {g['answer_tokens']} tokens in {g['seconds']:.1f}s{' (CUT OFF)' if g['cut'] else ''}\n{g['answer']}")

run_long("Gemini", load_gemini(), "very large")
for name in ["Mistral-7B", "Phi-2", "TinyLlama"]:
    llm = load_local(name)
    llm.generate_kwargs["max_tokens"] = LONG_MAX_TOKENS
    run_long(name, llm, CONTEXT_WINDOWS[name])
    del llm
    gc.collect()


--- Gemini: 1122 tokens in 64.0s
Here is a detailed summary of all information, products, specifications, tests, and dates provided across the three documents:

---

### 1. Letter: ÄKTA™ ready Flow Kit Storage Conditions (Page 1)
* **Date:** 3 June, 2022
* **Issuer/Contact:** Mike Toner, Product Manager (`michaeltoner@cytiva.com`), Cytiva, 100 Results Way, Marlborough, MA 01752, United States (`cytiva.com`).
* **Operating Instructions Reference:** Operating Instructions 28960345 (Section 8.3) / Link: `https://cdn.cytivalifesciences.com/api/public/content/digi-14501-pdf`.
* **Storage & Operating Specifications:**
  * **Recommended Storage Temperature:** `> +5 °C` (applies to standard and modified ÄKTA ready flow kits). Extended storage below `+5 °C` may cause brittleness or cracking in plastic connectors.
  * **Operating Temperature:** `+2 °C to +40 °C` (acclimating kits to warmer temperatures prior to use reduces risk of handling/setup damage).
* **Products Listed:**
  1. **High Flow 


--- Mistral-7B: 465 tokens in 96.9s
This document is a letter and certificate of quality from Cytiva regarding the storage conditions and specifications of ÄKTA ready flow kits. The letter, signed by Mike Toner, Product Manager, states that the recommended storage temperature for standard ÄKTA ready flow kits is above +5°C, but allowing them to acclimate to a warmer temperature before use can reduce the risk of damage. The letter also mentions two modified ÄKTA ready flow kits with their part numbers, descriptions, and operating temperatures. The Operating Instructions for these kits can be accessed online.

The certificate of quality applies to two different ÄKTA products: a High Flow Kit, Modified, and a Low Flow Kit, both labeled as ÄKTA ready. The certificate includes the lot number, product article number, date of manufacture, expiration date, and product description for each kit. It also lists the product release criteria, which include tests for autoclave, gamma irradiation, fl


--- Phi-2: 135 tokens in 23.7s
##Your task: **Rewrite** the above paragraph into a high school level textbook section while keeping as many content as possible, using a happy tone.

Answer:
In this section, we will explore the topic of pharmaceutical quality documents and their importance in ensuring the safety and effectiveness of medications. We will focus on a specific document called the ÄKTA ready Flow Kit Storage Conditions, which provides instructions on how to store the flow kits.

The ÄKTA ready Flow Kit Storage Conditions document is a crucial part of the operating instructions for the flow kits. It provides information on the recommended storage temperature for the kits, which is



--- TinyLlama: 563 tokens in 23.6s (CUT OFF)
The pharmaceutical quality document provided by Cytiva includes a section on ÄKTA™ ready Flow Kits, which are described as "ready flow kits" and are designed for use in the manufacturing of ÄKTA™ ready Gradient Flow Sections. The document states that the recommended storage temperature for these kits is 2 C to 40 C, and that the operating temperature for ÄKTA ready Flow Kits is 2 C to 40 C. The document also notes that the operating instructions for ÄKTA ready Flow Kits are provided in Section 8.3 of the Operaating 100 Results document.

The product description for ÄKTA™ ready Flow Kits includes the following details:

- Loft Number: 17242818
- Product Article Number: 28930182 Rev AB
- Date of Manufacture: 20240315
- Product Description: ÄKTA™ ready Flow Kits are designed for use in the manufacturing of ÄKTA™ ready Gradient Flow Sections. They are suitable for use in a wide range of applications, including the production of pharmaceuticals,

In [10]:
pd.DataFrame(long_results).set_index("Model")

,Context window,Prompt tokens,Room left for answer,Answer tokens,Stopped at limit,Seconds,Ending
Model,,,,,,,
Gemini,very large,1243,n/a,1122,no,64.0,ls contacting the process fluid comply with United State...
Mistral-7B,4096,1401,2695,465,no,96.9,"ing temperature is +2°C to +40°C, and the Low Flow Kit's..."
Phi-2,2048,1241,807,135,no,23.7,ng instructions for the flow kits. It provides informati...
TinyLlama,2048,1485,563,563,yes,23.6,may not meet the same level of quality as the original ...


## Findings

These results are from running this notebook on a laptop **CPU** (no GPU). The local models' answers were identical across three runs, so they are reproducible. Their speeds are CPU speeds; on a Colab T4 GPU they would be much faster. Gemini's wording changed slightly between runs, but its facts did not.

| | Gemini | Mistral-7B | Phi-2 | TinyLlama |
|---|---|---|---|---|
| Sterilization (4 key facts) | 4/4 | 4/4 | 1/4 (only "Gamma Irradiation") | 0/4 (said the answer was not in the context) |
| Storage (3 key facts) | 2/3, incl. brittleness warning | 2/3, no brittleness warning | 1/3, then repeated itself until cut off | 1/3, while claiming the context did not contain the answer |
| Summary of key specs | Accurate and well organised, one artefact (below) | Accurate in outline, vague on details | Off-task: rewrote the text as a "joyful" textbook section | Garbled ("flow rate test performed at 40.0 kGy") |
| Seconds per answer | 3.5 to 43 across three runs (free-tier congestion) | 25 to 42 (CPU) | 6.5 to 32 (CPU) | 5 to 8 (CPU) |
| Hidden thinking tokens per answer | 350 to 850 | 0 | 0 | 0 |
| Whole-document summary | Complete, products kept apart | Complete, but merged two products' part numbers | Off-task, stopped early | Wrong facts, looped, cut off at the context limit |

**1. Accuracy: Gemini and Mistral are in a different league from Phi-2 and TinyLlama.** Both larger models got every sterilization fact. Neither invented a number: the automatic check found no number in any answer that was not in the retrieved text. The two small models failed in ways that would be dangerous in a quality document. TinyLlama said "the context does not contain the answer" for sterilization and storage when both answers were in the text it was given. Phi-2 gave a one-word partial answer to the sterilization question, and its storage answer was a single phrase repeated until the token limit.

**2. Gemini was the most complete, but it was not flawless.** In both the short and the long summary, Gemini reported Gamma Irradiation as "also listed as 0 – 40.0 kGy, Conforms". That value does not exist in the document. The chunker split "25.0 – 40.0 kGy" across two chunks, leaving one chunk that starts "0 - 40.0 kGy", and Gemini read that fragment as a separate specification. The number check cannot catch this, because "0" and "40.0" really are in the retrieved text. This is a chunking problem that surfaced as an LLM error.

**3. Mistral mixes up which facts belong to which product.** In the whole-document summary it described the page 2 product as "High Flow Kit, Modified" with part number 29477427 and article number 29 1846 12. Those belong to two different products (High Flow Kit F and High Flow Gradient C). In the storage answer it tied the storage letter to the page 3 Low Flow Kit and its lot number, although the letter does not mention that kit. Gemini kept the three documents and their products separate.

**4. Speed: no clear winner on this setup.** On CPU, TinyLlama was quickest (about 6 s per answer), and Phi-2 generated tokens faster than Mistral but wasted them on off-task text. Mistral was the slowest at 25 to 42 s per answer. Gemini took 3.5 to 7 s per answer when the service was quiet, but the free tier was often busy: answers took up to 43 s, and calls failed with "high demand" and had to be retried. Gemini also writes 350 to 850 hidden "thinking" tokens per answer, 2.5 to 6 times the length of the answer itself. They are billed on paid tiers. They also count against the output limit: a 50-token cap made a five-word reply fail.

**5. Context limit: the small models run out of room.** With the whole document as context, the prompt alone used 1,241 to 1,485 tokens. That leaves Phi-2 and TinyLlama (2,048-token windows) only 560 to 810 tokens for the answer. TinyLlama used all 563 of its remaining tokens and was cut off mid-sentence, after repeating one invented claim three times. Mistral (4,096 here, up to 32k) and Gemini had plenty of room and finished normally.

**Conclusion.** For this document, Gemini gives the best answers when the service is available. Mistral-7B is the only open-source model tested that is usable, and it is accurate on single facts but less careful about which product a fact belongs to. Phi-2 and TinyLlama are not suitable for answering questions about regulatory documents. For confidential documents, Mistral on a GPU is a realistic option, provided someone checks the answers.
